In [1]:
import pandas as pd

In [2]:
import openpyxl
import re

In [3]:
wb = openpyxl.load_workbook('table.xlsx')

In [4]:
ws = wb.active

In [5]:
parsed_data = []

In [6]:
rows = list(ws.iter_rows(values_only=True))

In [7]:
data_rows = rows[26:]

In [8]:
for i in range(0, len(data_rows), 3):
    if i + 2 >= len(data_rows):
        break

    row1 = data_rows[i]
    row3 = data_rows[i + 2]

    qty = int(row1[14]) if row1[14] else 0
    amount = float(row1[30]) if row1[30] else 0
    
    messy_string = row3[0]

    if not messy_string.startswith('CN Code'):
        break
        
    CN_regex = re.search(r'CN Code:\s*(\d+)', messy_string)
    Net_regex = re.search(r'Net weight\s\(g\):([\d\s]+)', messy_string)
    Gross_regex = re.search(r'Gross weight\s\(g\):([\d\s]+)', messy_string)

    code = CN_regex.group(1) if CN_regex else None
    net_str = Net_regex.group(1) if Net_regex else None
    gross_str = Gross_regex.group(1) if Gross_regex else None

    net = float(re.sub(r'\s+', '', net_str))/1000
    gross = float(re.sub(r'\s+', '', gross_str))/1000

    parsed_data.append({
        'CN code': code,
        'qty': qty,
        'net': net,
        'gross': gross,
        'amount': amount,
    })

In [9]:
clean_df = pd.DataFrame(parsed_data)

In [10]:
clean_df.to_excel('clean_data.xlsx', index=False)

In [11]:
indexed = clean_df.set_index('CN code')

In [12]:
grouped = indexed.groupby(indexed.index)

In [13]:
compact = grouped.sum()

In [14]:
compact.loc['Total'] = compact.sum()

In [15]:
print(compact)

              qty      net    gross      amount
CN code                                        
30051000    150.0    0.540    0.540    23058.00
33041000   6610.0   29.051  106.525  1024868.53
33042000   9362.0   45.582  141.394  1982266.90
33043000    156.0    1.888    3.192    32467.68
33049100   1872.0   14.391   70.653   473683.80
33049900   4640.0   85.246  193.361  1199132.60
35061000    318.0    1.887    5.028    69913.32
67041900    320.0    1.280    3.460    63574.00
67042000      6.0    0.114    0.114     1965.54
82032000    288.0    7.884    7.884    61010.40
96033090    366.0    8.973    9.213    95713.68
Total     24088.0  196.836  541.364  5027654.45


In [16]:
compact.to_excel('calculated.xlsx')